In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# remove this
from pyspark.sql.functions import when, sum, col, size, split


# visualization
import seaborn as sns
import matplotlib.pyplot as plt

from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler

In [2]:
LABEL_COLUMN='price'
SEED=42

spark = SparkSession.builder \
    .appName("Airbnb") \
    .getOrCreate()

raw_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("multiLine", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .option("mode", "PERMISSIVE")
    .parquet("./total_data.parquet")
)

raw_df.createOrReplaceTempView('raw_listing')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/10 10:36:41 WARN Utils: Your hostname, thiago, resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface wlp0s20f3)
26/05/10 10:36:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 10:36:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/10 10:36:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [3]:
# as the dataset is has a lot of columns, we did an analysis and selected the most 
# good appearing to the prediction. 

selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification, 
        month,
        security_deposit,
        cleaning_fee
    FROM raw_listing
""")

selected_df.createOrReplaceTempView("selected_df")

## Initial Parsing

In [4]:
selected_df.show()

+-------------------+-------------------+-----------------+-------------------+------------------+------------+---------+--------+----+--------------------+---------+-----------------------------+--------------------------------+-----+----------------+------------+
|           latitude|          longitude|host_is_superhost|host_listings_count|     property_type|accommodates|bathrooms|bedrooms|beds|           amenities|    price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|
+-------------------+-------------------+-----------------+-------------------+------------------+------------+---------+--------+----+--------------------+---------+-----------------------------+--------------------------------+-----+----------------+------------+
|-22.965919031411442| -43.17896230586568|                f|                2.0|       Condominium|           5|      1.0|     2.0| 2.0|{TV,"Cable TV",In...|  $307.00|                            f|      

Shape of the dataset:

In [5]:
# change this to pure SQL
print(f'Rows: {selected_df.count()}, Columns: {len(selected_df.columns)}')

Rows: 784122, Columns: 16


Checking for Null's:

In [6]:
selected_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in selected_df.columns
]).show()

+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+
|       1|        1|              386|                386|            1|           1|     1494|     776|2335|        1|    1|                            2|                               2|    2|          361064|      269336|
+--------+---------+-----------------+-------------------+-------------+------------+---------+-----

As we can see there's a lot of missing values in the `security_deposit` and `cleaning_fee` columns.

Dropping them from the database seems to be a good decision.

In [7]:
rows_before_drop = selected_df.count()
selected_df = selected_df.drop('security_deposit', 'cleaning_fee')

In [8]:
selected_df = selected_df.dropna()
print(f"({selected_df.count()}, {len(selected_df.columns)}) - {rows_before_drop - selected_df.count()} rows dropped")

(780035, 14) - 4087 rows dropped


Checking if there is any missing left:

In [9]:
selected_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in selected_df.columns
]).show()


+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|       0|        0|                0|                  0|            0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+



In [10]:
selected_df.createOrReplaceTempView('selected_df')

### Checking data types

In [11]:
selected_df.printSchema()

root
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_listings_count: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- accommodates: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- beds: string (nullable = true)
 |-- amenities: string (nullable = true)
 |-- price: string (nullable = true)
 |-- require_guest_profile_picture: string (nullable = true)
 |-- require_guest_phone_verification: string (nullable = true)
 |-- month: string (nullable = true)



TODO: A brief explanation of what need to be changed

### Checking data types

In [12]:
selected_df = spark.sql("""
    SELECT
        TRY_CAST(latitude AS DOUBLE) AS latitude,
        TRY_CAST(longitude AS DOUBLE) AS longitude,
        TRY_CAST(host_is_superhost AS BOOLEAN) AS host_is_superhost,
        TRY_CAST(TRY_CAST(host_listings_count AS DOUBLE) AS INT) AS host_listings_count,
        property_type,
        TRY_CAST(TRY_CAST(accommodates AS DOUBLE) AS INT) AS accommodates,
        TRY_CAST(TRY_CAST(bathrooms AS DOUBLE) AS INT) AS bathrooms,
        TRY_CAST(TRY_CAST(bedrooms AS DOUBLE) AS INT) AS bedrooms,
        TRY_CAST(TRY_CAST(beds AS DOUBLE) AS INT) AS beds,
        amenities,
        TRY_CAST(REGEXP_REPLACE(price, '[\\\\$,]', '') AS DOUBLE) AS price,
        TRY_CAST(require_guest_profile_picture AS BOOLEAN) AS require_guest_profile_picture,
        TRY_CAST(require_guest_phone_verification AS BOOLEAN) AS require_guest_phone_verification, 
        TRY_CAST(month AS INT) AS month
    FROM selected_df
""")

selected_df.createOrReplaceTempView("selected_df")

selected_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in selected_df.columns
]).show()

+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|       0|        0|                0|                  0|            0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+



## EDA + Data Cleaning

### Dealing with outliers

In [13]:
def get_max_fence(column):
    qt = selected_df.approxQuantile(column, [0.25,0.75], 0.01)
    q1 = qt[0]
    upper = qt[1]
    iqr = upper - q1
    max_fence = upper + 1.5 * iqr
    return max_fence

#### `host_listing_count`

In [14]:
listing_count_max_fence = get_max_fence('host_listings_count')
print(listing_count_max_fence)

6.0


In [15]:
rows_before = selected_df.count()
selected_df = selected_df.filter(col('host_listings_count') <= listing_count_max_fence)
print(f"{rows_before - selected_df.count()} rows were deleted.")

99528 rows were deleted.


In [16]:
selected_df = selected_df.withColumn(
    'host_listings_count',
    when(col('host_listings_count') == 0.0, 1.0).otherwise(col('host_listings_count'))
)

#### `price`

In [17]:
price_max_fence = get_max_fence('price')
print(price_max_fence)

1274.5


In [18]:
rows_before = selected_df.count()
selected_df = selected_df.filter(col('price') <= price_max_fence)
print(f"{rows_before - selected_df.count()} rows were deleted.")

66373 rows were deleted.


#### `property_type`

In [19]:
categories_to_append = ('Aparthotel', 'Earth house', 'Chalet', 'Cottage', 'Tiny house',
                        'Boutique hotel', 'Hotel', 'Casa particular (Cuba)', 'Bungalow',
                        'Nature lodge', 'Cabin', 'Castle', 'Treehouse', 'Island', 'Boat', 'Tent',
                        'Resort', 'Hut', 'Campsite', 'Barn', 'Dorm', 'Camper/RV', 'Farm stay', 'Yurt',
                        'Tipi', 'Pension (South Korea)', 'Dome house', 'Igloo', 'Casa particular',
                        'Houseboat', 'Lighthouse', 'Plane', 'Train', 'Parking Space')

selected_df = selected_df.withColumn(
    'property_type',
    when(col('property_type').isin(*categories_to_append), 'Other')
    .otherwise(col('property_type'))
)

#### `beds`

In [20]:
beds_max_fence = get_max_fence('beds')
print(beds_max_fence)

6.0


In [21]:
rows_before = selected_df.count()
selected_df = selected_df.filter(col('beds') <= beds_max_fence)
print(f"{rows_before - selected_df.count()} rows were deleted.")

13846 rows were deleted.


#### `amenities`

In [22]:
selected_df = selected_df.withColumn('n_amenities', size(split(col('amenities'), ',')) + 1)
mode_row = (selected_df
            .filter(col('amenities') != '{}')
            .groupBy('n_amenities')
            .count()
            .orderBy(col('count').desc())
            .first())
mode_value = mode_row['n_amenities']
selected_df = selected_df.withColumn(
    'n_amenities',
    when(col('amenities') == '{}', mode_value).otherwise(col('n_amenities'))
)
# 4. Remover a coluna original 'amenities'
selected_df = selected_df.drop('amenities')

In [23]:
amenities_max_fence = get_max_fence('n_amenities')
print(amenities_max_fence)

40.5


In [24]:
rows_before = selected_df.count()
selected_df = selected_df.filter(col('n_amenities') <= amenities_max_fence)
print(f"{rows_before - selected_df.count()} rows were deleted.")

22771 rows were deleted.


## Encoding

In [25]:
df_label_encoder = selected_df 

bool_columns = [
    'host_is_superhost', 
    'require_guest_profile_picture', 
    'require_guest_phone_verification'
]

# idk if this is necessary as the columns is already true/false
# please check
for column in bool_columns:
    df_label_encoder = df_label_encoder.withColumn(
        column,
        when(col(column) == 't', 1)
        .when(col(column) == 'f', 0)
        .otherwise(col(column).cast('integer')) 
    )
indexer = StringIndexer(inputCol='property_type', outputCol='property_type_encoded')
df_label_encoder = indexer.fit(df_label_encoder).transform(df_label_encoder)
df_label_encoder = (df_label_encoder
                    .drop('property_type')
                    .withColumnRenamed('property_type_encoded', 'property_type'))
print('Columns encoded')

Columns encoded


## Model

In [26]:
from pyspark.ml.regression import DecisionTreeRegressor
models = [
    (
        "Linear Regression",
        LinearRegression(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            maxIter=50,
            regParam=0.1,
            elasticNetParam=0.0
        )
    ),
    (
        "Decision Tree Regressor",
        DecisionTreeRegressor(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            seed=SEED
        )
    ),
    (
        "Random Forest Regressor",
        RandomForestRegressor(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            numTrees=30,
            maxDepth=8,
            maxBins=64,
            seed=SEED
        )
    )
]

In [27]:
feature_cols = [
    "host_is_superhost",
    "host_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "month",
    "property_type",
    "require_guest_profile_picture",
    "require_guest_phone_verification",
    "n_amenities"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
prepared_df = assembler.transform(df_label_encoder)

train_prepared, test_prepared = prepared_df.randomSplit([0.8, 0.2], seed=SEED)

print(f"Linhas no Treino: {train_prepared.count()}")
print(f"Linhas no Teste: {test_prepared.count()}")

Linhas no Treino: 461755


Linhas no Teste: 115762


In [28]:
def evaluate_model(model_name: str, predictions) -> None:
    predictions = predictions.cache()
    predictions.createOrReplaceTempView("predictions")

    rmse = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="rmse",
    ).evaluate(predictions)
    mae = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="mae",
    ).evaluate(predictions)
    r2 = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="r2",
    ).evaluate(predictions)

    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R2: {r2:.6f}")

    predictions.unpersist()

In [29]:
for model_name, estimator in models:
    model = estimator.fit(train_prepared)
    predictions = model.transform(test_prepared)
    evaluate_model(model_name, predictions)

RMSE: 240.33
MAE: 179.84
R2: 0.294057


RMSE: 239.60
MAE: 177.80
R2: 0.298358


26/05/10 10:39:57 WARN DAGScheduler: Broadcasting large task binary with size 1404.9 KiB


RMSE: 228.18
MAE: 168.74
R2: 0.363625
